# Brain Tumor Classification — CT & MRI
### Comparative Study: EfficientNetB3 | ResNet50 | VGG16 | Vision Transformer

> **Structure:** Each model has its own fully independent cells for training, confusion matrix, ROC-AUC, F1/Precision/Recall bar chart, and GradCAM. Nothing is merged.

## 1 · Environment Setup

In [ ]:
!pip install timm torchmetrics scikit-learn opencv-python -q

In [ ]:
import os, random, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

import timm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


## 2 · Configuration

In [ ]:
# ── Sanity check: verify dataset structure BEFORE training ──
from torchvision.datasets import ImageFolder

_check_ct  = ImageFolder("/kaggle/input/datasets/murtozalikhon/brain-tumor-multimodal-image-ct-and-mri/Dataset/Brain Tumor CT scan Images")
_check_mri = ImageFolder("/kaggle/input/datasets/murtozalikhon/brain-tumor-multimodal-image-ct-and-mri/Dataset/Brain Tumor MRI images")

print("CT  classes:", _check_ct.classes,  "→ count:", len(_check_ct.classes),  "| total images:", len(_check_ct))
print("MRI classes:", _check_mri.classes, "→ count:", len(_check_mri.classes), "| total images:", len(_check_mri))

In [ ]:
DATA_DIR_CT  = "/kaggle/input/datasets/murtozalikhon/brain-tumor-multimodal-image-ct-and-mri/Dataset/Brain Tumor CT scan Images"
DATA_DIR_MRI = "/kaggle/input/datasets/murtozalikhon/brain-tumor-multimodal-image-ct-and-mri/Dataset/Brain Tumor MRI images"

MODALITY    = 'MRI'          # 'CT' or 'MRI'
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 1e-4
NUM_CLASSES = 2
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.15
PATIENCE    = 7

os.makedirs('./results', exist_ok=True)
print(f"Modality : {MODALITY}")


## 3 · Data Loading & Preprocessing

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATA_DIR = DATA_DIR_CT if MODALITY == 'CT' else DATA_DIR_MRI

# ── FIX: Three separate ImageFolder instances — transforms are isolated per split ──
# Avoids shared-dataset transform mutation (data leakage / augmentation suppression).
_base_dataset = ImageFolder(root=DATA_DIR)
CLASS_NAMES   = _base_dataset.classes
total         = len(_base_dataset)
test_size     = int(total * TEST_SPLIT)
val_size      = int(total * VAL_SPLIT)
train_size    = total - val_size - test_size

# Reproducible index split
generator   = torch.Generator().manual_seed(42)
all_indices = torch.randperm(total, generator=generator).tolist()
train_indices = all_indices[:train_size]
val_indices   = all_indices[train_size : train_size + val_size]
test_indices  = all_indices[train_size + val_size :]

# Each split owns its dataset — transform is locked, can never be overwritten
_train_ds = ImageFolder(root=DATA_DIR, transform=train_transforms)
_val_ds   = ImageFolder(root=DATA_DIR, transform=eval_transforms)
_test_ds  = ImageFolder(root=DATA_DIR, transform=eval_transforms)

train_data = Subset(_train_ds, train_indices)
val_data   = Subset(_val_ds,   val_indices)
test_data  = Subset(_test_ds,  test_indices)

# Class counts from full dataset (train-only would be cleaner but difference is tiny)
label_counts = pd.Series(_base_dataset.targets).value_counts().sort_index()

print(f"Classes : {CLASS_NAMES}")
print(f"Total   : {total}  |  Train : {train_size}  |  Val : {val_size}  |  Test : {test_size}")


In [ ]:
# ── Data Augmentation Visualization & Class Balance ──────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

# ── 1. Class distribution BEFORE and AFTER split ─────────────────
train_labels = [_train_ds.targets[i] for i in train_indices]
val_labels   = [_val_ds.targets[i]   for i in val_indices]
test_labels  = [_test_ds.targets[i]  for i in test_indices]
all_labels   = _base_dataset.targets

def count_classes(labels, class_names):
    counts = [labels.count(i) for i in range(len(class_names))]
    return counts

full_counts  = count_classes(list(all_labels),  CLASS_NAMES)
train_counts = count_classes(train_labels,       CLASS_NAMES)
val_counts   = count_classes(val_labels,         CLASS_NAMES)
test_counts  = count_classes(test_labels,        CLASS_NAMES)

fig = plt.figure(figsize=(18, 12))
fig.suptitle(f'Data Augmentation & Class Balance — {MODALITY}', fontsize=15, fontweight='bold')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Class distribution across splits ──────────────────────
ax1   = fig.add_subplot(gs[0, 0])
x     = np.arange(len(CLASS_NAMES))
w     = 0.2
bars1 = ax1.bar(x - 1.5*w, full_counts,  w, label='Full',  color='steelblue')
bars2 = ax1.bar(x - 0.5*w, train_counts, w, label='Train', color='seagreen')
bars3 = ax1.bar(x + 0.5*w, val_counts,   w, label='Val',   color='darkorange')
bars4 = ax1.bar(x + 1.5*w, test_counts,  w, label='Test',  color='firebrick')
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(int(bar.get_height())), ha='center', fontsize=8)
ax1.set_xticks(x)
ax1.set_xticklabels(CLASS_NAMES)
ax1.set_title('Class Distribution Across Splits')
ax1.set_ylabel('Image Count')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# ── Plot 2: Pie chart — full dataset balance ──────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.pie(full_counts, labels=CLASS_NAMES, autopct='%1.1f%%',
        colors=['steelblue', 'firebrick'], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title(f'Full Dataset Balance\n(Total: {sum(full_counts)} images)')

# ── Plot 3: Original vs Augmented images side by side ────────────
ax3 = fig.add_subplot(gs[1, :])
ax3.axis('off')
ax3.set_title('Original  vs  Augmented (same image, different augmentations)',
              fontsize=11, pad=10)

# Grab one image from train set (no transform — raw PIL)
from torchvision.datasets import ImageFolder
_raw_ds = ImageFolder(root=DATA_DIR)  # no transform
raw_img, raw_lbl = _raw_ds[0]         # PIL Image

_mean = np.array([0.485, 0.456, 0.406])
_std  = np.array([0.229, 0.224, 0.225])

# Show 1 original + 5 augmented versions
n_aug  = 5
inner  = gridspec.GridSpecFromSubplotSpec(1, n_aug + 1, subplot_spec=gs[1, :], wspace=0.05)

# Original
ax_orig = fig.add_subplot(inner[0])
orig_tensor = eval_transforms(raw_img)
orig_np     = (orig_tensor.permute(1,2,0).numpy() * _std + _mean).clip(0,1)
ax_orig.imshow(orig_np)
ax_orig.set_title('Original', fontsize=9, color='steelblue', fontweight='bold')
ax_orig.axis('off')

# Augmented versions
for j in range(n_aug):
    ax_aug = fig.add_subplot(inner[j + 1])
    aug_tensor = train_transforms(raw_img)
    aug_np     = (aug_tensor.permute(1,2,0).numpy() * _std + _mean).clip(0,1)
    ax_aug.imshow(aug_np)
    ax_aug.set_title(f'Aug #{j+1}', fontsize=9, color='seagreen')
    ax_aug.axis('off')

plt.savefig('./results/augmentation_balance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary print ─────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Dataset Balance Summary — {MODALITY}")
print(f"{'='*55}")
print(f"  {'Split':<10} {'Total':>7}  " + "  ".join(f"{c:>10}" for c in CLASS_NAMES))
print(f"  {'-'*53}")
for split, counts in [('Full', full_counts), ('Train', train_counts),
                       ('Val',  val_counts),  ('Test',  test_counts)]:
    print(f"  {split:<10} {sum(counts):>7}  " + "  ".join(f"{c:>10}" for c in counts))
print(f"{'='*55}")
print(f"\n  Augmentations applied to TRAIN only:")
print(f"     RandomHorizontalFlip (p=0.5)")
print(f"     RandomVerticalFlip   (p=0.2)")
print(f"     RandomRotation       (±15°)")
print(f"     ColorJitter          (brightness & contrast ±0.2)")
print(f"     RandomAffine         (translate ±10%)")
print(f"  Val & Test: NO augmentation (eval_transforms only)")

In [ ]:
# Sample images visualization
def show_samples(dataset, class_names, n=8):
    # FIX: Do NOT set dataset.dataset.transform — each split already has the right transform locked.
    loader = DataLoader(dataset, batch_size=n, shuffle=True)
    images, labels = next(iter(loader))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    fig.suptitle(f'Sample Images — {MODALITY}', fontsize=13)
    for i, ax in enumerate(axes.flat):
        img = (images[i] * std + mean).permute(1,2,0).numpy().clip(0,1)
        ax.imshow(img)
        ax.set_title(class_names[labels[i]], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('./results/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

show_samples(train_data, CLASS_NAMES)


## 4 · Model Builder, Trainer & Shared Utilities



In [ ]:
def build_model(model_name, num_classes):
    if model_name == 'efficientnet':
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        for param in model.features[:6].parameters():
            param.requires_grad = False
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(in_f, 512),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        for name, param in model.named_parameters():
            if not any(l in name for l in ['layer3', 'layer4', 'fc']):
                param.requires_grad = False
        in_f = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(in_f, 512),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
    elif model_name == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        for param in model.features.parameters():
            param.requires_grad = False
        for param in model.features[24:].parameters():
            param.requires_grad = True
        model.classifier = nn.Sequential(
            nn.Linear(512*7*7, 4096), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(4096, 1024),    nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(1024, num_classes)
        )
    elif model_name == 'vit':
        model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
        for block in list(model.blocks.children())[:-4]:
            for param in block.parameters():
                param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"  Trainable : {trainable:,} / {total_p:,}  ({100*trainable/total_p:.1f}%)")
    return model.to(DEVICE)

In [ ]:
def get_loaders():
    tl = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    vl = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tl, vl, te

In [ ]:
# ── GradCAM class & helpers ───────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        # FIX: register_full_backward_hook (register_backward_hook is deprecated)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, m, i, o):
        out = o[0] if isinstance(o, tuple) else o
        self.activations = out.detach()

    def _save_gradient(self, m, gi, go):
        g = go[0]
        if g is not None:
            self.gradients = g.detach()

    def generate(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, class_idx].backward()

        acts  = self.activations
        grads = self.gradients

        if acts is None or grads is None:
            print("[GradCAM] Warning: hooks did not fire. Returning blank map.")
            return np.zeros((14, 14)), class_idx

        ndim = acts.ndim

        if ndim == 3:
            # FIX: ViT path — tokens are (1, num_tokens, embed_dim)
            # Drop CLS token at index 0, then pool over embed dim
            acts  = acts[:, 1:, :]
            grads = grads[:, 1:, :]
            weights = grads.mean(dim=2, keepdim=True)            # (1, patches, 1)
            cam     = torch.relu((weights * acts).sum(dim=2))    # (1, patches)
            num_patches = cam.shape[1]
            grid        = int(num_patches ** 0.5)
            cam = cam.reshape(grid, grid).cpu().numpy()
        elif ndim == 4:
            # CNN path — feature maps are (1, C, H, W)
            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam     = torch.relu((weights * acts).sum(dim=1))
            cam     = cam.squeeze(0).cpu().numpy()
        else:
            print(f"[GradCAM] Unexpected activation ndim={ndim}. Returning blank map.")
            return np.zeros((14, 14)), class_idx

        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam, class_idx


def get_target_layer(model, model_name):
    if model_name == 'efficientnet': return model.features[-1]
    if model_name == 'resnet50':     return model.layer4[-1]
    if model_name == 'vgg16':        return model.features[-1]
    if model_name == 'vit':          return model.blocks[-1].norm1


# ── GradCAM visualization helper (shared by all models) ─────────
import cv2

def plot_gradcam(model, model_label, save_dir):
    import cv2
    _mean = np.array([0.485, 0.456, 0.406])
    _std  = np.array([0.229, 0.224, 0.225])
    arch  = model_label.lower().replace(' ', '_').replace('(', '').replace(')', '')
    # map label to arch key
    arch_key = 'vit' if 'vit' in arch else 'efficientnet' if 'efficientnet' in arch else                'resnet50' if 'resnet' in arch else 'vgg16'

    # FIX: test_data already has eval_transforms — no mutation needed
    gcam_loader = DataLoader(test_data, batch_size=8, shuffle=True)
    gcam_imgs, gcam_lbls = next(iter(gcam_loader))
    gcam_imgs = gcam_imgs[:4].to(DEVICE)
    gcam_lbls = gcam_lbls[:4]

    gradcam = GradCAM(model, get_target_layer(model, arch_key))

    fig, axes = plt.subplots(4, 3, figsize=(12, 16))
    fig.suptitle(f'GradCAM — {model_label} ({MODALITY})', fontsize=13)

    for i in range(4):
        cam_map, pred_idx = gradcam.generate(gcam_imgs[i:i+1])
        img_np = (gcam_imgs[i].cpu().permute(1,2,0).numpy() * _std + _mean).clip(0,1)
        cam_r  = cv2.resize(cam_map, (IMG_SIZE, IMG_SIZE))
        overlay = (0.6 * img_np + 0.4 * plt.cm.jet(cam_r)[:,:,:3]).clip(0,1)

        axes[i,0].imshow(img_np)
        axes[i,0].set_title(f'Input  True: {CLASS_NAMES[gcam_lbls[i]]}', fontsize=9)
        axes[i,0].axis('off')

        axes[i,1].imshow(cam_r, cmap='jet')
        axes[i,1].set_title(f'GradCAM  Pred: {CLASS_NAMES[pred_idx]}', fontsize=9)
        axes[i,1].axis('off')

        axes[i,2].imshow(overlay)
        axes[i,2].set_title('Overlay', fontsize=9)
        axes[i,2].axis('off')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/gradcam.png", dpi=150, bbox_inches='tight')
    plt.show()


# Storage for cross-model comparison
all_results = {}
print("All shared utilities ready.")


In [ ]:
# ── train_model & evaluate — shared by all four models ────────────────
# train_model('<arch>') and the final comparison cell calls evaluate().

def evaluate(model, loader, criterion):
    """Run one evaluation pass; return loss, acc (%), preds, labels, probs."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, lbls)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            total_loss += loss.item() * imgs.size(0)
            correct += (preds == lbls).sum().item()
            total += imgs.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(lbls.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
    return (total_loss / total,
            100.0 * correct / total,
            np.array(all_preds),
            np.array(all_labels),
            np.array(all_probs))


def train_model(model_name):
    """
    Build, train, and evaluate a model.
    Returns:
        model, history, te_preds, te_labels, te_probs,
        te_acc, best_val_acc, save_dir
    """
    print(f"\n{'='*60}")
    print(f" Training : {model_name.upper()} | Modality : {MODALITY}")
    print(f"{'='*60}")
    
    save_dir = f"./results/{model_name}_{MODALITY}"
    os.makedirs(save_dir, exist_ok=True)

    # ── Data loaders (splits were fixed in Section 3) ──────────────
    train_loader, val_loader, test_loader = get_loaders()

    # ── Class-weighted loss (computed from TRAIN split only) ───────
    train_labels_arr = np.array([_train_ds.targets[i] for i in train_indices])
    class_counts = np.bincount(train_labels_arr, minlength=NUM_CLASSES).astype(float)
    class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float).to(DEVICE)
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = build_model(model_name, NUM_CLASSES)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )

    history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc', 'lr']}
    best_val_acc = 0.0
    patience_ctr = 0
    best_path = f"{save_dir}/best_model.pth"

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        
        for imgs, lbls in tqdm(train_loader, 
                               desc=f"Epoch {epoch:02d}/{EPOCHS} [{model_name}]",
                               leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            tr_loss += loss.item() * imgs.size(0)
            tr_correct += (logits.argmax(1) == lbls).sum().item()
            tr_total += imgs.size(0)

        tr_loss /= tr_total
        tr_acc = 100.0 * tr_correct / tr_total

        # ── Validate ───────────────────────────────────────────────
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_acc)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        print(f" Epoch {epoch:02d}/{EPOCHS} "
              f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.2f}% "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}%")

        # ── Early stopping & checkpoint ────────────────────────────
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_ctr = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f" Early stopping at epoch {epoch}.")
                break

    # ── Load best weights & evaluate on test set ───────────────────
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    te_loss, te_acc, te_preds, te_labels, te_probs = evaluate(
        model, test_loader, criterion
    )

    print(f"\n Best val acc : {best_val_acc:.2f}%")
    print(f" Test acc     : {te_acc:.2f}%")
    print(f" Test loss    : {te_loss:.4f}")

    return (model, history, te_preds, te_labels, te_probs, 
            te_acc, best_val_acc, save_dir)


print("train_model() and evaluate() are ready.")

---
## 5 · EfficientNetB3



### 5.1 · EfficientNetB3 — Training

In [ ]:
# ── EfficientNetB3: Train ──────────────────────────────────────────
model_efficientnet, history_efficientnet, te_preds_efficientnet, te_labels_efficientnet, te_probs_efficientnet, \
    te_acc_efficientnet, best_val_acc_efficientnet, save_dir_efficientnet = train_model('efficientnet')


### 5.2 · EfficientNetB3 — Classification Report

In [ ]:
# ── EfficientNetB3: Classification Report ──────────────────────────
print(f"\n  Classification Report — EfficientNetB3:")
print(classification_report(te_labels_efficientnet, te_preds_efficientnet, target_names=CLASS_NAMES, digits=4))

prec_efficientnet, rec_efficientnet, f1_efficientnet, _ = precision_recall_fscore_support(
    te_labels_efficientnet, te_preds_efficientnet, labels=range(NUM_CLASSES), average='weighted'
)
print(f"  Weighted Precision : {prec_efficientnet:.4f}")
print(f"  Weighted Recall    : {rec_efficientnet:.4f}")
print(f"  Weighted F1        : {f1_efficientnet:.4f}")


### 5.3 · EfficientNetB3 — Training Curves

In [ ]:
# ── EfficientNetB3: Training Curves ────────────────────────────────
n = len(history_efficientnet['train_loss'])
x = range(1, n+1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle(f'Training History — EfficientNetB3 ({MODALITY})', fontsize=13)

axes[0].plot(x, history_efficientnet['train_loss'], color='steelblue', label='Train', lw=1.5)
axes[0].plot(x, history_efficientnet['val_loss'],   color='firebrick', label='Val',   lw=1.5)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x, history_efficientnet['train_acc'], color='steelblue', label='Train', lw=1.5)
axes[1].plot(x, history_efficientnet['val_acc'],   color='firebrick', label='Val',   lw=1.5)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(x, history_efficientnet['lr'], color='seagreen', lw=1.5)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_dir_efficientnet}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


### 5.4 · EfficientNetB3 — Confusion Matrix

In [ ]:
# ── EfficientNetB3: Confusion Matrix ───────────────────────────────
cm_efficientnet     = confusion_matrix(te_labels_efficientnet, te_preds_efficientnet)
cm_pct_efficientnet = cm_efficientnet.astype(float) / cm_efficientnet.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Confusion Matrix — EfficientNetB3 ({MODALITY})', fontsize=13)

sns.heatmap(cm_efficientnet, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Sample Counts')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_pct_efficientnet, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Row-Normalized (%)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f"{save_dir_efficientnet}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


### 5.5 · EfficientNetB3 — ROC-AUC Curve

In [ ]:
# ── EfficientNetB3: ROC-AUC Curve ──────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

if NUM_CLASSES == 2:
    fpr_efficientnet, tpr_efficientnet, _ = roc_curve(te_labels_efficientnet, te_probs_efficientnet[:, 1])
    roc_auc_efficientnet = auc(fpr_efficientnet, tpr_efficientnet)
    ax.plot(fpr_efficientnet, tpr_efficientnet, color='steelblue', lw=2,
            label=f'ROC curve  (AUC = {roc_auc_efficientnet:.4f})')
else:
    y_bin_efficientnet = label_binarize(te_labels_efficientnet, classes=range(NUM_CLASSES))
    colors_ = ['steelblue','firebrick','seagreen','darkorange']
    roc_auc_efficientnet = 0.0
    for i, cls in enumerate(CLASS_NAMES):
        fpr_i, tpr_i, _ = roc_curve(y_bin_efficientnet[:, i], te_probs_efficientnet[:, i])
        auc_i = auc(fpr_i, tpr_i)
        roc_auc_efficientnet += auc_i
        ax.plot(fpr_i, tpr_i, color=colors_[i], lw=2, label=f'{cls}  (AUC = {auc_i:.4f})')
    roc_auc_efficientnet /= NUM_CLASSES

ax.plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — EfficientNetB3 ({MODALITY})')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{save_dir_efficientnet}/roc_auc.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"  EfficientNetB3 AUC: {roc_auc_efficientnet:.4f}")


### 5.6 · EfficientNetB3 — Precision / Recall / F1 Bar Chart

In [ ]:
# ── EfficientNetB3: Per-Class Precision / Recall / F1 ─────────────
prec_cls_efficientnet, rec_cls_efficientnet, f1_cls_efficientnet, sup_efficientnet = precision_recall_fscore_support(
    te_labels_efficientnet, te_preds_efficientnet, labels=range(NUM_CLASSES)
)
x_     = np.arange(len(CLASS_NAMES))
width_ = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_ - width_, prec_cls_efficientnet, width_, label='Precision', color='steelblue')
ax.bar(x_,          rec_cls_efficientnet,  width_, label='Recall',    color='seagreen')
ax.bar(x_ + width_, f1_cls_efficientnet,   width_, label='F1-Score',  color='firebrick')

ax.set_xticks(x_)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Per-Class Metrics — EfficientNetB3 ({MODALITY})')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for i in range(len(CLASS_NAMES)):
    ax.text(i - width_, prec_cls_efficientnet[i] + 0.02, f'{prec_cls_efficientnet[i]:.2f}', ha='center', fontsize=8)
    ax.text(i,          rec_cls_efficientnet[i]  + 0.02, f'{rec_cls_efficientnet[i]:.2f}',  ha='center', fontsize=8)
    ax.text(i + width_, f1_cls_efficientnet[i]   + 0.02, f'{f1_cls_efficientnet[i]:.2f}',   ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{save_dir_efficientnet}/metrics_bar.png", dpi=150, bbox_inches='tight')
plt.show()


### 5.7 · EfficientNetB3 — GradCAM Visualization

In [ ]:
# ── EfficientNetB3: GradCAM ─────────────────────────────────────────────
plot_gradcam(model_efficientnet, 'EfficientNetB3', save_dir_efficientnet)


### 5.8 · EfficientNetB3 — Save Summary & Free GPU Memory

In [ ]:
# ── EfficientNetB3: Save summary & release GPU ─────────────────────
all_results['efficientnet'] = {
    'Model':        'efficientnet',
    'Modality':     MODALITY,
    'Best_Val_Acc': round(best_val_acc_efficientnet, 4),
    'Test_Acc':     round(te_acc_efficientnet, 4),
    'Precision':    round(prec_efficientnet, 4),
    'Recall':       round(rec_efficientnet, 4),
    'F1_Score':     round(f1_efficientnet, 4),
    'ROC_AUC':      round(roc_auc_efficientnet, 4),
    'Epochs_Run':   len(history_efficientnet['train_loss'])
}
pd.DataFrame([all_results['efficientnet']]).to_csv(
    f"{save_dir_efficientnet}/summary.csv", index=False
)
del model_efficientnet
torch.cuda.empty_cache()
print(f"  EfficientNetB3 results saved to {save_dir_efficientnet}")


---
## 6 · ResNet50



### 6.1 · ResNet50 — Training

In [ ]:
# ── ResNet50: Train ──────────────────────────────────────────
model_resnet50, history_resnet50, te_preds_resnet50, te_labels_resnet50, te_probs_resnet50, \
    te_acc_resnet50, best_val_acc_resnet50, save_dir_resnet50 = train_model('resnet50')


### 6.2 · ResNet50 — Classification Report

In [ ]:
# ── ResNet50: Classification Report ──────────────────────────
print(f"\n  Classification Report — ResNet50:")
print(classification_report(te_labels_resnet50, te_preds_resnet50, target_names=CLASS_NAMES, digits=4))

prec_resnet50, rec_resnet50, f1_resnet50, _ = precision_recall_fscore_support(
    te_labels_resnet50, te_preds_resnet50, labels=range(NUM_CLASSES), average='weighted'
)
print(f"  Weighted Precision : {prec_resnet50:.4f}")
print(f"  Weighted Recall    : {rec_resnet50:.4f}")
print(f"  Weighted F1        : {f1_resnet50:.4f}")


### 6.3 · ResNet50 — Training Curves

In [ ]:
# ── ResNet50: Training Curves ────────────────────────────────
n = len(history_resnet50['train_loss'])
x = range(1, n+1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle(f'Training History — ResNet50 ({MODALITY})', fontsize=13)

axes[0].plot(x, history_resnet50['train_loss'], color='steelblue', label='Train', lw=1.5)
axes[0].plot(x, history_resnet50['val_loss'],   color='firebrick', label='Val',   lw=1.5)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x, history_resnet50['train_acc'], color='steelblue', label='Train', lw=1.5)
axes[1].plot(x, history_resnet50['val_acc'],   color='firebrick', label='Val',   lw=1.5)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(x, history_resnet50['lr'], color='seagreen', lw=1.5)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_dir_resnet50}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


### 6.4 · ResNet50 — Confusion Matrix

In [ ]:
# ── ResNet50: Confusion Matrix ───────────────────────────────
cm_resnet50     = confusion_matrix(te_labels_resnet50, te_preds_resnet50)
cm_pct_resnet50 = cm_resnet50.astype(float) / cm_resnet50.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Confusion Matrix — ResNet50 ({MODALITY})', fontsize=13)

sns.heatmap(cm_resnet50, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Sample Counts')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_pct_resnet50, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Row-Normalized (%)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f"{save_dir_resnet50}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


### 6.5 · ResNet50 — ROC-AUC Curve

In [ ]:
# ── ResNet50: ROC-AUC Curve ──────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

if NUM_CLASSES == 2:
    fpr_resnet50, tpr_resnet50, _ = roc_curve(te_labels_resnet50, te_probs_resnet50[:, 1])
    roc_auc_resnet50 = auc(fpr_resnet50, tpr_resnet50)
    ax.plot(fpr_resnet50, tpr_resnet50, color='steelblue', lw=2,
            label=f'ROC curve  (AUC = {roc_auc_resnet50:.4f})')
else:
    y_bin_resnet50 = label_binarize(te_labels_resnet50, classes=range(NUM_CLASSES))
    colors_ = ['steelblue','firebrick','seagreen','darkorange']
    roc_auc_resnet50 = 0.0
    for i, cls in enumerate(CLASS_NAMES):
        fpr_i, tpr_i, _ = roc_curve(y_bin_resnet50[:, i], te_probs_resnet50[:, i])
        auc_i = auc(fpr_i, tpr_i)
        roc_auc_resnet50 += auc_i
        ax.plot(fpr_i, tpr_i, color=colors_[i], lw=2, label=f'{cls}  (AUC = {auc_i:.4f})')
    roc_auc_resnet50 /= NUM_CLASSES

ax.plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — ResNet50 ({MODALITY})')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{save_dir_resnet50}/roc_auc.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"  ResNet50 AUC: {roc_auc_resnet50:.4f}")


### 6.6 · ResNet50 — Precision / Recall / F1 Bar Chart

In [ ]:
# ── ResNet50: Per-Class Precision / Recall / F1 ─────────────
prec_cls_resnet50, rec_cls_resnet50, f1_cls_resnet50, sup_resnet50 = precision_recall_fscore_support(
    te_labels_resnet50, te_preds_resnet50, labels=range(NUM_CLASSES)
)
x_     = np.arange(len(CLASS_NAMES))
width_ = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_ - width_, prec_cls_resnet50, width_, label='Precision', color='steelblue')
ax.bar(x_,          rec_cls_resnet50,  width_, label='Recall',    color='seagreen')
ax.bar(x_ + width_, f1_cls_resnet50,   width_, label='F1-Score',  color='firebrick')

ax.set_xticks(x_)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Per-Class Metrics — ResNet50 ({MODALITY})')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for i in range(len(CLASS_NAMES)):
    ax.text(i - width_, prec_cls_resnet50[i] + 0.02, f'{prec_cls_resnet50[i]:.2f}', ha='center', fontsize=8)
    ax.text(i,          rec_cls_resnet50[i]  + 0.02, f'{rec_cls_resnet50[i]:.2f}',  ha='center', fontsize=8)
    ax.text(i + width_, f1_cls_resnet50[i]   + 0.02, f'{f1_cls_resnet50[i]:.2f}',   ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{save_dir_resnet50}/metrics_bar.png", dpi=150, bbox_inches='tight')
plt.show()


### 6.7 · ResNet50 — GradCAM Visualization

In [ ]:
# ── ResNet50: GradCAM ─────────────────────────────────────────────
plot_gradcam(model_resnet50, 'ResNet50', save_dir_resnet50)


### 6.8 · ResNet50 — Save Summary & Free GPU Memory

In [ ]:
# ── ResNet50: Save summary & release GPU ─────────────────────
all_results['resnet50'] = {
    'Model':        'resnet50',
    'Modality':     MODALITY,
    'Best_Val_Acc': round(best_val_acc_resnet50, 4),
    'Test_Acc':     round(te_acc_resnet50, 4),
    'Precision':    round(prec_resnet50, 4),
    'Recall':       round(rec_resnet50, 4),
    'F1_Score':     round(f1_resnet50, 4),
    'ROC_AUC':      round(roc_auc_resnet50, 4),
    'Epochs_Run':   len(history_resnet50['train_loss'])
}
pd.DataFrame([all_results['resnet50']]).to_csv(
    f"{save_dir_resnet50}/summary.csv", index=False
)
del model_resnet50
torch.cuda.empty_cache()
print(f"  ResNet50 results saved to {save_dir_resnet50}")


---
## 7 · VGG16



### 7.1 · VGG16 — Training

In [ ]:
# ── VGG16: Train ──────────────────────────────────────────
model_vgg16, history_vgg16, te_preds_vgg16, te_labels_vgg16, te_probs_vgg16, \
    te_acc_vgg16, best_val_acc_vgg16, save_dir_vgg16 = train_model('vgg16')


### 7.2 · VGG16 — Classification Report

In [ ]:
# ── VGG16: Classification Report ──────────────────────────
print(f"\n  Classification Report — VGG16:")
print(classification_report(te_labels_vgg16, te_preds_vgg16, target_names=CLASS_NAMES, digits=4))

prec_vgg16, rec_vgg16, f1_vgg16, _ = precision_recall_fscore_support(
    te_labels_vgg16, te_preds_vgg16, labels=range(NUM_CLASSES), average='weighted'
)
print(f"  Weighted Precision : {prec_vgg16:.4f}")
print(f"  Weighted Recall    : {rec_vgg16:.4f}")
print(f"  Weighted F1        : {f1_vgg16:.4f}")


### 7.3 · VGG16 — Training Curves

In [ ]:
# ── VGG16: Training Curves ────────────────────────────────
n = len(history_vgg16['train_loss'])
x = range(1, n+1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle(f'Training History — VGG16 ({MODALITY})', fontsize=13)

axes[0].plot(x, history_vgg16['train_loss'], color='steelblue', label='Train', lw=1.5)
axes[0].plot(x, history_vgg16['val_loss'],   color='firebrick', label='Val',   lw=1.5)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x, history_vgg16['train_acc'], color='steelblue', label='Train', lw=1.5)
axes[1].plot(x, history_vgg16['val_acc'],   color='firebrick', label='Val',   lw=1.5)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(x, history_vgg16['lr'], color='seagreen', lw=1.5)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_dir_vgg16}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


### 7.4 · VGG16 — Confusion Matrix

In [ ]:
# ── VGG16: Confusion Matrix ───────────────────────────────
cm_vgg16     = confusion_matrix(te_labels_vgg16, te_preds_vgg16)
cm_pct_vgg16 = cm_vgg16.astype(float) / cm_vgg16.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Confusion Matrix — VGG16 ({MODALITY})', fontsize=13)

sns.heatmap(cm_vgg16, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Sample Counts')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_pct_vgg16, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Row-Normalized (%)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f"{save_dir_vgg16}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


### 7.5 · VGG16 — ROC-AUC Curve

In [ ]:
# ── VGG16: ROC-AUC Curve ──────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

if NUM_CLASSES == 2:
    fpr_vgg16, tpr_vgg16, _ = roc_curve(te_labels_vgg16, te_probs_vgg16[:, 1])
    roc_auc_vgg16 = auc(fpr_vgg16, tpr_vgg16)
    ax.plot(fpr_vgg16, tpr_vgg16, color='steelblue', lw=2,
            label=f'ROC curve  (AUC = {roc_auc_vgg16:.4f})')
else:
    y_bin_vgg16 = label_binarize(te_labels_vgg16, classes=range(NUM_CLASSES))
    colors_ = ['steelblue','firebrick','seagreen','darkorange']
    roc_auc_vgg16 = 0.0
    for i, cls in enumerate(CLASS_NAMES):
        fpr_i, tpr_i, _ = roc_curve(y_bin_vgg16[:, i], te_probs_vgg16[:, i])
        auc_i = auc(fpr_i, tpr_i)
        roc_auc_vgg16 += auc_i
        ax.plot(fpr_i, tpr_i, color=colors_[i], lw=2, label=f'{cls}  (AUC = {auc_i:.4f})')
    roc_auc_vgg16 /= NUM_CLASSES

ax.plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — VGG16 ({MODALITY})')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{save_dir_vgg16}/roc_auc.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"  VGG16 AUC: {roc_auc_vgg16:.4f}")


### 7.6 · VGG16 — Precision / Recall / F1 Bar Chart

In [ ]:
# ── VGG16: Per-Class Precision / Recall / F1 ─────────────
prec_cls_vgg16, rec_cls_vgg16, f1_cls_vgg16, sup_vgg16 = precision_recall_fscore_support(
    te_labels_vgg16, te_preds_vgg16, labels=range(NUM_CLASSES)
)
x_     = np.arange(len(CLASS_NAMES))
width_ = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_ - width_, prec_cls_vgg16, width_, label='Precision', color='steelblue')
ax.bar(x_,          rec_cls_vgg16,  width_, label='Recall',    color='seagreen')
ax.bar(x_ + width_, f1_cls_vgg16,   width_, label='F1-Score',  color='firebrick')

ax.set_xticks(x_)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Per-Class Metrics — VGG16 ({MODALITY})')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for i in range(len(CLASS_NAMES)):
    ax.text(i - width_, prec_cls_vgg16[i] + 0.02, f'{prec_cls_vgg16[i]:.2f}', ha='center', fontsize=8)
    ax.text(i,          rec_cls_vgg16[i]  + 0.02, f'{rec_cls_vgg16[i]:.2f}',  ha='center', fontsize=8)
    ax.text(i + width_, f1_cls_vgg16[i]   + 0.02, f'{f1_cls_vgg16[i]:.2f}',   ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{save_dir_vgg16}/metrics_bar.png", dpi=150, bbox_inches='tight')
plt.show()


### 7.7 · VGG16 — GradCAM Visualization

In [ ]:
# ── VGG16: GradCAM ─────────────────────────────────────────────
plot_gradcam(model_vgg16, 'VGG16', save_dir_vgg16)


### 7.8 · VGG16 — Save Summary & Free GPU Memory

In [ ]:
# ── VGG16: Save summary & release GPU ─────────────────────
all_results['vgg16'] = {
    'Model':        'vgg16',
    'Modality':     MODALITY,
    'Best_Val_Acc': round(best_val_acc_vgg16, 4),
    'Test_Acc':     round(te_acc_vgg16, 4),
    'Precision':    round(prec_vgg16, 4),
    'Recall':       round(rec_vgg16, 4),
    'F1_Score':     round(f1_vgg16, 4),
    'ROC_AUC':      round(roc_auc_vgg16, 4),
    'Epochs_Run':   len(history_vgg16['train_loss'])
}
pd.DataFrame([all_results['vgg16']]).to_csv(
    f"{save_dir_vgg16}/summary.csv", index=False
)
del model_vgg16
torch.cuda.empty_cache()
print(f"  VGG16 results saved to {save_dir_vgg16}")


---
## 8 · ViT (Vision Transformer)



### 8.1 · ViT (Vision Transformer) — Training

In [ ]:
# ── ViT (Vision Transformer): Train ──────────────────────────────────────────
model_vit, history_vit, te_preds_vit, te_labels_vit, te_probs_vit, \
    te_acc_vit, best_val_acc_vit, save_dir_vit = train_model('vit')


### 8.2 · ViT (Vision Transformer) — Classification Report

In [ ]:
# ── ViT (Vision Transformer): Classification Report ──────────────────────────
print(f"\n  Classification Report — ViT (Vision Transformer):")
print(classification_report(te_labels_vit, te_preds_vit, target_names=CLASS_NAMES, digits=4))

prec_vit, rec_vit, f1_vit, _ = precision_recall_fscore_support(
    te_labels_vit, te_preds_vit, labels=range(NUM_CLASSES), average='weighted'
)
print(f"  Weighted Precision : {prec_vit:.4f}")
print(f"  Weighted Recall    : {rec_vit:.4f}")
print(f"  Weighted F1        : {f1_vit:.4f}")


### 8.3 · ViT (Vision Transformer) — Training Curves

In [ ]:
# ── ViT (Vision Transformer): Training Curves ────────────────────────────────
n = len(history_vit['train_loss'])
x = range(1, n+1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle(f'Training History — ViT (Vision Transformer) ({MODALITY})', fontsize=13)

axes[0].plot(x, history_vit['train_loss'], color='steelblue', label='Train', lw=1.5)
axes[0].plot(x, history_vit['val_loss'],   color='firebrick', label='Val',   lw=1.5)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x, history_vit['train_acc'], color='steelblue', label='Train', lw=1.5)
axes[1].plot(x, history_vit['val_acc'],   color='firebrick', label='Val',   lw=1.5)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(x, history_vit['lr'], color='seagreen', lw=1.5)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_dir_vit}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


### 8.4 · ViT (Vision Transformer) — Confusion Matrix

In [ ]:
# ── ViT (Vision Transformer): Confusion Matrix ───────────────────────────────
cm_vit     = confusion_matrix(te_labels_vit, te_preds_vit)
cm_pct_vit = cm_vit.astype(float) / cm_vit.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Confusion Matrix — ViT (Vision Transformer) ({MODALITY})', fontsize=13)

sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Sample Counts')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_pct_vit, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Row-Normalized (%)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f"{save_dir_vit}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


### 8.5 · ViT (Vision Transformer) — ROC-AUC Curve

In [ ]:
# ── ViT (Vision Transformer): ROC-AUC Curve ──────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

if NUM_CLASSES == 2:
    fpr_vit, tpr_vit, _ = roc_curve(te_labels_vit, te_probs_vit[:, 1])
    roc_auc_vit = auc(fpr_vit, tpr_vit)
    ax.plot(fpr_vit, tpr_vit, color='steelblue', lw=2,
            label=f'ROC curve  (AUC = {roc_auc_vit:.4f})')
else:
    y_bin_vit = label_binarize(te_labels_vit, classes=range(NUM_CLASSES))
    colors_ = ['steelblue','firebrick','seagreen','darkorange']
    roc_auc_vit = 0.0
    for i, cls in enumerate(CLASS_NAMES):
        fpr_i, tpr_i, _ = roc_curve(y_bin_vit[:, i], te_probs_vit[:, i])
        auc_i = auc(fpr_i, tpr_i)
        roc_auc_vit += auc_i
        ax.plot(fpr_i, tpr_i, color=colors_[i], lw=2, label=f'{cls}  (AUC = {auc_i:.4f})')
    roc_auc_vit /= NUM_CLASSES

ax.plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — ViT (Vision Transformer) ({MODALITY})')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{save_dir_vit}/roc_auc.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"  ViT (Vision Transformer) AUC: {roc_auc_vit:.4f}")


### 8.6 · ViT (Vision Transformer) — Precision / Recall / F1 Bar Chart

In [ ]:
# ── ViT (Vision Transformer): Per-Class Precision / Recall / F1 ─────────────
prec_cls_vit, rec_cls_vit, f1_cls_vit, sup_vit = precision_recall_fscore_support(
    te_labels_vit, te_preds_vit, labels=range(NUM_CLASSES)
)
x_     = np.arange(len(CLASS_NAMES))
width_ = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_ - width_, prec_cls_vit, width_, label='Precision', color='steelblue')
ax.bar(x_,          rec_cls_vit,  width_, label='Recall',    color='seagreen')
ax.bar(x_ + width_, f1_cls_vit,   width_, label='F1-Score',  color='firebrick')

ax.set_xticks(x_)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Per-Class Metrics — ViT (Vision Transformer) ({MODALITY})')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for i in range(len(CLASS_NAMES)):
    ax.text(i - width_, prec_cls_vit[i] + 0.02, f'{prec_cls_vit[i]:.2f}', ha='center', fontsize=8)
    ax.text(i,          rec_cls_vit[i]  + 0.02, f'{rec_cls_vit[i]:.2f}',  ha='center', fontsize=8)
    ax.text(i + width_, f1_cls_vit[i]   + 0.02, f'{f1_cls_vit[i]:.2f}',   ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{save_dir_vit}/metrics_bar.png", dpi=150, bbox_inches='tight')
plt.show()


### 8.7 · ViT (Vision Transformer) — GradCAM Visualization

In [ ]:
# ── ViT (Vision Transformer): GradCAM ─────────────────────────────────────────────
plot_gradcam(model_vit, 'ViT (Vision Transformer)', save_dir_vit)


### 8.8 · ViT (Vision Transformer) — Save Summary & Free GPU Memory

In [ ]:
# ── ViT (Vision Transformer): Save summary & release GPU ─────────────────────
all_results['vit'] = {
    'Model':        'vit',
    'Modality':     MODALITY,
    'Best_Val_Acc': round(best_val_acc_vit, 4),
    'Test_Acc':     round(te_acc_vit, 4),
    'Precision':    round(prec_vit, 4),
    'Recall':       round(rec_vit, 4),
    'F1_Score':     round(f1_vit, 4),
    'ROC_AUC':      round(roc_auc_vit, 4),
    'Epochs_Run':   len(history_vit['train_loss'])
}
pd.DataFrame([all_results['vit']]).to_csv(
    f"{save_dir_vit}/summary.csv", index=False
)
del model_vit
torch.cuda.empty_cache()
print(f"  ViT (Vision Transformer) results saved to {save_dir_vit}")


---
## 9 · Cross-Model Comparison



In [ ]:
# ── Comparison dataframe ─────────────────────────────────────
compare_df = pd.DataFrame(list(all_results.values()))
compare_df = compare_df.sort_values('Test_Acc', ascending=False).reset_index(drop=True)
print("\nFinal Comparison — All Models:")
print(compare_df[['Model','Test_Acc','Best_Val_Acc','Precision',
                   'Recall','F1_Score','ROC_AUC','Epochs_Run']].to_string(index=False))
compare_df.to_csv('./results/model_comparison.csv', index=False)


In [ ]:
# ── Grouped bar chart: Test Acc / F1 / AUC ───────────────────
metrics_     = ['Test_Acc', 'F1_Score', 'ROC_AUC']
bar_colors   = ['steelblue', 'firebrick', 'seagreen']
x_c          = np.arange(len(compare_df))
width_c      = 0.25
model_labels = [m.upper() for m in compare_df['Model']]

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, color) in enumerate(zip(metrics_, bar_colors)):
    vals = compare_df[metric].values.copy()
    if metric == 'Test_Acc':
        vals = vals / 100.0
    bars = ax.bar(x_c + (i - 1) * width_c, vals, width_c,
                  label=metric.replace('_', ' '), color=color)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x_c)
ax.set_xticklabels(model_labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Model Comparison — Accuracy / F1 / AUC  ({MODALITY})', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./results/comparison_grouped_bar.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Overlay ROC curves for all 4 models ──────────────────────
MODEL_NAMES = ['efficientnet', 'resnet50', 'vgg16', 'vit']
palette     = ['steelblue', 'firebrick', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(8, 7))

for model_name, color in zip(MODEL_NAMES, palette):
    save_dir_ = f"./results/{model_name}_{MODALITY}"
    best_path = f"{save_dir_}/best_model.pth"
    m = build_model(model_name, NUM_CLASSES)
    m.load_state_dict(torch.load(best_path, map_location=DEVICE))
    # FIX: test_data already uses eval_transforms via its own _test_ds — no mutation needed
    te_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    # FIX: use train-only label counts (not full dataset) to avoid leaking test/val distribution
    _train_lbls_arr = np.array([_train_ds.targets[i] for i in train_indices])
    class_counts  = np.bincount(_train_lbls_arr, minlength=NUM_CLASSES).astype(float)
    class_weights = torch.tensor(1.0/class_counts, dtype=torch.float).to(DEVICE)
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES
    criterion_    = nn.CrossEntropyLoss(weight=class_weights)
    _, _, _, labels_, probs_ = evaluate(m, te_loader, criterion_)
    fpr_, tpr_, _ = roc_curve(labels_, probs_[:, 1] if NUM_CLASSES == 2 else probs_[:, 0])
    roc_auc_      = auc(fpr_, tpr_)
    ax.plot(fpr_, tpr_, color=color, lw=2,
            label=f'{model_name.upper()}  (AUC = {roc_auc_:.4f})')
    del m
    torch.cuda.empty_cache()

ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — All Models ({MODALITY})', fontsize=13)
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./results/roc_all_models.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Final summary table ───────────────────────────────────────
print(f"\n{'='*70}")
print(f"  FINAL RESULTS — {MODALITY} — All Models")
print(f"{'='*70}")
print(f"  {'Model':<20} {'Test Acc':>10} {'F1':>8} {'AUC':>8} {'Precision':>10} {'Recall':>8}")
print(f"  {'-'*66}")
for _, row in compare_df.iterrows():
    print(f"  {row['Model']:<20} "
          f"{row['Test_Acc']:>9.2f}%  "
          f"{row['F1_Score']:>7.4f}  "
          f"{row['ROC_AUC']:>7.4f}  "
          f"{row['Precision']:>9.4f}  "
          f"{row['Recall']:>7.4f}")
print(f"{'='*70}")
print(f"\nAll outputs saved under ./results/")
